# (Reference solutions) 01_returns_covariance_and_linear_algebra

> This is the **full reference-solution edition** of the matching main notebook. Try the exercises yourself first, then compare. All explanations and answers are original to this project.

# Week 1 — Returns, Risk and Linear Algebra

> Part of the open-source teaching project **quant-math-roadmap**.
> For **education and research methodology only** — not investment advice; no result here represents a profitable or investable strategy.

## Learning objectives

- Correctly compute simple returns, log returns and cumulative returns.
- Compute annualized mean, annualized volatility, the covariance matrix and the correlation matrix.
- Compute portfolio variance with the quadratic form $w^\top\Sigma w$.
- Understand eigenvalues, eigenvectors and positive semidefinite (PSD) matrices.

## Estimated study time

About 8–10 hours.

## Prerequisites

- Vector and matrix operations
- Definitions of mean and variance

## External resources

- [MIT OpenCourseWare 18.06SC Linear Algebra](https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/)
- [NTU OpenCourseWare: Foundations of Financial Literacy](https://ocw.aca.ntu.edu.tw/courses/110S204)

> External resources are linked for reference only; this project does not reproduce any copyrighted course material.

In [ ]:
# Teaching style setup (deterministic look, consistent figures)
import matplotlib as _mpl
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## Concepts

### Returns

Given prices $P_t$, the **simple return** and the **log return** are defined as:

$$ r_t = \frac{P_t - P_{t-1}}{P_{t-1}}, \qquad \ell_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right). $$

Log returns are **additive over time**: the multi-period log return equals the sum of the per-period ones.

### Portfolio variance

With a weight vector $w$ and covariance matrix $\Sigma$, portfolio variance is the quadratic form:

$$ \operatorname{Var}(r_p) = w^\top \Sigma\, w. $$

Because a variance can never be negative, $w^\top\Sigma w \ge 0$ for every $w$ — which is exactly what it means for $\Sigma$ to be **positive semidefinite (PSD)**, equivalent to all of its eigenvalues being $\ge 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.data import SyntheticConfig, generate_correlated_prices
from quant_math_roadmap.finance.returns import simple_returns, log_returns
from quant_math_roadmap.finance.metrics import (
    annualized_mean, annualized_volatility,
    covariance_matrix, correlation_matrix,
)
from quant_math_roadmap.finance.portfolio import equal_weights, portfolio_variance
from quant_math_roadmap.math.linear_algebra import (
    eigendecomposition, is_positive_semidefinite,
)

config = SyntheticConfig(n_assets=4, n_periods=756, seed=11,
                         average_correlation=0.4)
prices = generate_correlated_prices(config)
prices.tail()

### Computing returns and comparing simple vs log

In [ ]:
simple = simple_returns(prices)
log = log_returns(prices)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(simple.index, simple.iloc[:, 0], label='simple return')
ax.plot(log.index, log.iloc[:, 0], label='log return')
ax.set_title(f'{prices.columns[0]}: simple vs log return')
ax.set_xlabel('Date')
ax.set_ylabel('Daily return')
ax.legend()
plt.show()

At the daily scale the two nearly overlap; the difference only becomes visible for larger returns. The advantage of log returns is additivity, which is very convenient for the regression and time-series models later on.

### Annualized mean and volatility (note: annualization is an explicit assumption)

In [ ]:
ann_mean = annualized_mean(simple, frequency='daily')
ann_vol = annualized_volatility(simple, frequency='daily')
summary = ann_mean.to_frame('Annualized mean').join(ann_vol.to_frame('Annualized volatility'))
summary

We pass `frequency='daily'` (252 trading days per year) **explicitly**. If the data were actually weekly but annualized with 252, volatility would be overstated by roughly $\sqrt{252/52}\approx 2.2$ times.

### Covariance, correlation matrix and the quadratic form

In [ ]:
cov = covariance_matrix(simple)
corr = correlation_matrix(simple)
print('Covariance matrix:')
print(cov.round(6))
print('\nCorrelation matrix:')
print(corr.round(3))

In [ ]:
weights = equal_weights(prices.shape[1])
# Compute the quadratic form w^T Sigma w by hand
manual = float(weights @ cov.to_numpy() @ weights)
# Using the reusable function
via_function = portfolio_variance(weights, cov.to_numpy())
print(f'Portfolio variance by hand     = {manual:.8f}')
print(f'Portfolio variance via function = {via_function:.8f}')
assert np.isclose(manual, via_function)

### Eigenvalues and the PSD check

In [ ]:
eigenvalues, eigenvectors = eigendecomposition(cov.to_numpy())
print('Eigenvalues of the covariance matrix:', np.round(eigenvalues, 8))
print('Is it PSD (all eigenvalues >= 0)?', is_positive_semidefinite(cov.to_numpy()))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(eigenvalues) + 1), eigenvalues)
ax.set_title('Eigenvalues of the covariance matrix')
ax.set_xlabel('Eigenvalue index')
ax.set_ylabel('Eigenvalue')
plt.show()

All eigenvalues are $\ge 0$, so the covariance matrix is PSD. The eigenvector belonging to the largest eigenvalue is often interpreted as the dominant direction of common movement across assets (related to PCA).

### What if the estimated matrix is not PSD?

Sometimes noise or floating-point error gives a covariance estimate a tiny negative eigenvalue. `nearest_psd()` clips negative eigenvalues to 0 (or some lower bound epsilon), projecting onto the nearest PSD matrix. This is a teaching-grade repair, not a substitute for shrinkage.

In [ ]:
from quant_math_roadmap.math.linear_algebra import nearest_psd

# Deliberately inject a small negative eigenvalue into the covariance matrix
noisy = cov.to_numpy().copy()
noisy[0, 0] -= 2 * eigenvalues.max()  # force a negative eigenvalue
print('Smallest eigenvalue before repair:', round(float(np.linalg.eigvalsh(noisy).min()), 6))
repaired = nearest_psd(noisy, epsilon=1e-8)
print('Smallest eigenvalue after repair:', round(float(np.linalg.eigvalsh(repaired).min()), 8))

## Exercises

Work through these in order. **Basic exercises** consolidate the definitions, **applied exercises** are hands-on coding, and the **reflection question** connects the mathematics to backtesting and research methodology.

> The main notebook ships runnable starter code for each coding exercise. Full reference answers live in the matching `_solution` notebook under `notebooks/en/solutions/`.

### Basic exercises

1. Explain in words why log returns are additive while simple returns are not.
2. Explain what impossible situation a covariance matrix with one negative eigenvalue would imply.
3. Why is the diagonal of a correlation matrix always 1?

### Applied exercises

In [ ]:
# Applied exercise 1: implement simple returns from scratch with numpy (no pct_change),
# and compare against the result of simple_returns().
p = prices.iloc[:, 0].to_numpy()
my_simple = (p[1:] - p[:-1]) / p[:-1]
print('Max error:', np.max(np.abs(my_simple - simple.iloc[:, 0].to_numpy())))

In [ ]:
# Applied exercise 2: compare the equal-weight portfolio variance with the variance of
# "buying only the lowest-volatility asset". Which is lower? Why does diversification usually help?
eq_var = portfolio_variance(equal_weights(prices.shape[1]), cov.to_numpy())
lowest_vol_idx = int(np.argmin(np.diag(cov.to_numpy())))
single_var = cov.to_numpy()[lowest_vol_idx, lowest_vol_idx]
print(f'Equal-weight variance = {eq_var:.8f}')
print(f'Lowest-volatility single-asset variance = {single_var:.8f}')
print('Diversification exploits correlations below 1 to lower the overall variance.')

### Reflection question

1. The covariance matrix is estimated from **historical** data. What can go wrong if you use it directly to predict **future** portfolio risk? What does that imply for backtesting?

## Quiz (self-check)
Answer the multiple-choice questions, then run the next cell to check yourself. Answers are stored as hashes, not plaintext.

**Q1. What is the matrix formula for portfolio variance?**
- A. wᵀΣw
- B. wᵀμ
- C. Σw
- D. wwᵀ

**Q2. Why must a covariance matrix be PSD?**
- A. Because it is a symmetric matrix
- B. Because the variance wᵀΣw of any portfolio can never be negative
- C. Because eigenvalues must be integers
- D. For numerical stability

**Q3. What is the key property of log returns versus simple returns?**
- A. Always larger
- B. Additive across periods
- C. Independent of prices
- D. Always positive

**Q4. To annualize daily volatility, multiply by?**
- A. 252
- B. √252
- C. 12
- D. √12

In [ ]:
my_answers = {1: 'A', 2: 'B', 3: 'B', 4: 'B'}

import hashlib as _hashlib
_expected = {1: '6bd2b1bae6008812', 2: 'bcd9ed8e382c41c5', 3: 'e703c3c3c9cc6729', 4: '21bfb83f4954b4fd'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: unanswered')
        continue
    _h = _hashlib.sha256(f'qmr-w1-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ correct' if _ok else '✘ incorrect'))
print(f'Score: {_n_correct} / {len(my_answers)}')

### Explanations

- **Q1 → A**: The quadratic form of the weight vector w with the covariance matrix Σ is the portfolio variance.
- **Q2 → B**: If a negative eigenvalue existed, you could construct a portfolio with negative variance — which is mathematically impossible.
- **Q3 → B**: The multi-period log return is the sum of the per-period ones; simple returns compound multiplicatively and cannot simply be added.
- **Q4 → B**: Under the i.i.d. assumption variance scales linearly with the horizon, so the standard deviation is multiplied by √252.

## Common mistakes

- **Forgetting that a return series has one fewer observation than the price series (there is no return on day one).**
- **Annualizing without stating the data frequency, assuming everything is daily.**
- **Calling an eigendecomposition on a non-symmetric or non-square matrix.**
- **Treating the sample covariance matrix as exact and ignoring that it is only a noisy estimate.**

## After this week, you should be able to

- [ ] Correctly compute simple/log returns and explain the difference.
- [ ] Compute annualized mean and volatility, and state the annualization assumption.
- [ ] Compute portfolio variance with $w^\top\Sigma w$.
- [ ] Check whether a matrix is PSD and explain what that means.

## References and attribution

- Every explanation, example and exercise in this notebook is **original** to this project.
- Recommended external resources: [`docs/resources.md`](../../../docs/resources.md).
- Concept notes: [`docs/math/`](../../../docs/math/) and [`docs/finance/`](../../../docs/finance/).

### Privacy and disclaimer

- This notebook contains no real personal information.
- This notebook uses only reproducible synthetic data and needs no network access.
- This notebook makes no claim of real-world trading profitability.